# Prompt Evals

In [27]:
# Load env variables and create client

from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [28]:
# Helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params ["system"] = system
    
    message = client.messages.create(**params)
    
    return message.content[0].text

In [29]:
# Function to generate a new dataset

import json

def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
                "format": "json" or "python" or "regex"
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    
    text = chat(messages, stop_sequences=["```"])
    
    return json.loads(text)

In [30]:
dataset = generate_dataset()

dataset

[{'task': 'Parse an AWS S3 bucket name and key from an S3 URI (e.g., s3://my-bucket/path/to/file.txt) using a regular expression',
  'format': 'regex'},
 {'task': 'Create a JSON CloudFormation template snippet that defines an AWS Lambda function with basic execution role and environment variables',
  'format': 'json'},
 {'task': 'Write a Python function that takes an AWS CloudWatch log timestamp in milliseconds since epoch and converts it to a human-readable ISO 8601 format string',
  'format': 'python'}]

In [31]:
# Generate the dataset and write it to 'dataset.json'

from pathlib import Path

out_path = Path("output/dataset.json")
out_path.parent.mkdir(parents=True, exist_ok=True)


with out_path.open('w') as f:
    json.dump(dataset, f, indent=2)

In [10]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""

    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    
    # TODO - refine the output format
    output = chat(messages)

    return output

In [ ]:
# running a single test case and grading the result 

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [12]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [13]:
# execute the eval pipeline

with open("output/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [14]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket ARN Region Extractor\n\nHere's a Python function that extracts the AWS region from an S3 bucket ARN:\n\n```python\ndef extract_region_from_s3_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket ARN string.\n    \n    S3 bucket ARNs don't typically contain region information in the standard format.\n    This function handles both standard S3 ARNs and extended ARNs (with region).\n    \n    Args:\n        arn (str): The S3 bucket ARN string\n        \n    Returns:\n        str: The AWS region, or 'us-east-1' as default if not specified\n        \n    Examples:\n        >>> extract_region_from_s3_arn('arn:aws:s3:::my-bucket')\n        'us-east-1'\n        >>> extract_region_from_s3_arn('arn:aws:s3:us-west-2::my-bucket')\n        'us-west-2'\n        >>> extract_region_from_s3_arn('arn:aws:s3:::')\n        'us-east-1'\n    \"\"\"\n    default_region = 'us-east-1'\n    \n    # Validate ARN format\n    if not isinstance(arn, s

## TODO

1. the grading system: replace the hardcoded score of 10 with an actual evaluation logic
2. output format: implement specific formatting instructions to change Claude verbose responses

## Iteration 1

In [17]:
# Function to grade a test case + output using a model

def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])

    return json.loads(eval_text)

In [18]:
# Passes a test case into Claude

def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)

    return output

In [19]:
# Function to execute a single test case and grade the output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [24]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [25]:
with open("output/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.666666666666667


In [26]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket ARN Region Extractor\n\nHere's a comprehensive solution:\n\n```python\ndef extract_region_from_s3_arn(arn: str) -> str:\n    \"\"\"\n    Extract the AWS region from an S3 bucket ARN string.\n    \n    S3 bucket ARNs have the format: arn:aws:s3:::bucket-name\n    Note: S3 bucket ARNs don't typically contain region info, so 'us-east-1' is default.\n    However, this function handles S3 object ARNs which may include region.\n    \n    Args:\n        arn (str): The S3 bucket or object ARN string\n        \n    Returns:\n        str: The AWS region, or 'us-east-1' if not specified\n        \n    Examples:\n        >>> extract_region_from_s3_arn('arn:aws:s3:::my-bucket')\n        'us-east-1'\n        >>> extract_region_from_s3_arn('arn:aws:s3:us-west-2:123456789012:bucket/my-bucket')\n        'us-west-2'\n    \"\"\"\n    # Default region\n    default_region = 'us-east-1'\n    \n    # Validate ARN format\n    if not arn or not isinstance(arn, str):\n      

## Iteration 2

In [32]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [33]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [34]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [35]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [36]:
with open("output/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.5


In [37]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef parse_s3_uri(uri):\n    pattern = r'^s3://([a-z0-9.-]+)/(.+)$'\n    match = re.match(pattern, uri)\n    if match:\n        return {\n            'bucket': match.group(1),\n            'key': match.group(2)\n        }\n    return None\n\n# Test\nprint(parse_s3_uri('s3://my-bucket/path/to/file.txt'))\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name and key from an S3 URI (e.g., s3://my-bucket/path/to/file.txt) using a regular expression",
      "format": "regex"
    },
    "score": 8.0,
    "reasoning": "The solution correctly addresses the core task of parsing S3 URIs using regex and returns properly structured output. However, the bucket name pattern is overly restrictive compared to AWS S3 naming rules, and the implementation lacks validation for edge cases. A more robust pattern would be `r'^s3://([a-zA-Z0-9._-]+)/(.+)$'` and could include optional validation for minimum bucket length and S3 naming constraints."
  },
  {
    "o

## Iteration 3 / Exercise!

Give the Model Grader more context on what a good solution looks like

In [ ]:
# Function to generate a new dataset

import json

def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
                "format": "json" or "python" or "regex",
                "solution_criteria": "Brief on what characteristics a good solution must include" 
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """

    # alternative for "solution_criteria": "Key criteria for evaluating the solution"

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    
    text = chat(messages, stop_sequences=["```"])
    
    return json.loads(text)

In [39]:
dataset = generate_dataset()

dataset

[{'task': 'Create a JSON configuration file for an AWS Lambda function that processes S3 events, with environment variables for bucket name and log level',
  'format': 'json',
  'solution_criteria': 'Valid JSON structure containing Lambda function configuration, S3 event source mapping, and environment variables section with at least bucket name and log level keys'},
 {'task': 'Write a Python function that validates an AWS IAM role ARN format and returns True if valid, False otherwise',
  'format': 'python',
  'solution_criteria': 'Function should validate ARN format (arn:aws:iam::account-id:role/role-name), handle edge cases, and return a boolean value'},
 {'task': 'Create a regular expression that matches valid AWS S3 bucket names according to AWS naming rules (3-63 characters, lowercase letters, numbers, hyphens, no consecutive hyphens, no hyphens at start/end)',
  'format': 'regex',
  'solution_criteria': 'Regex pattern should correctly validate bucket name constraints: length betw

In [40]:
# Generate the dataset and write it to 'dataset.json'

from pathlib import Path

out_path = Path("output/dataset.json")
out_path.parent.mkdir(parents=True, exist_ok=True)


with out_path.open('w') as f:
    json.dump(dataset, f, indent=2)

In [41]:
# Add 

def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Solution Criteria:
<solution_criteria>
{test_case["solution_criteria"]}
</solution_criteria>


Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])

    return json.loads(eval_text)

In [42]:
with open("output/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.0


In [43]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport json\n\nlambda_config = {\n    \"FunctionName\": \"S3EventProcessor\",\n    \"Runtime\": \"python3.11\",\n    \"Role\": \"arn:aws:iam::ACCOUNT_ID:role/lambda-s3-role\",\n    \"Handler\": \"index.handler\",\n    \"Timeout\": 60,\n    \"MemorySize\": 256,\n    \"Environment\": {\n        \"Variables\": {\n            \"BUCKET_NAME\": \"my-s3-bucket\",\n            \"LOG_LEVEL\": \"INFO\"\n        }\n    },\n    \"EventInvokeConfig\": {\n        \"DestinationConfig\": {\n            \"OnSuccess\": {\n                \"Type\": \"SNS\",\n                \"Destination\": \"arn:aws:sns:us-east-1:ACCOUNT_ID:success-topic\"\n            },\n            \"OnFailure\": {\n                \"Type\": \"SNS\",\n                \"Destination\": \"arn:aws:sns:us-east-1:ACCOUNT_ID:failure-topic\"\n            }\n        }\n    }\n}\n\nwith open('lambda_config.json', 'w') as f:\n    json.dump(lambda_config, f, indent=2)\n\nprint(json.dumps(lambda_config, indent=2))\n",
    "